# DINOv2-Base Diverse-Frame Pruning on MELD

This post-extraction notebook loads the merged 20% DINOv2-base diverse-frame embeddings bundled in the `benchmark-code` Kaggle dataset, evaluates them with the previous MLP, and retrains a matched MLP. No video or VLM extraction is repeated.

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
CHECKPOINT = CODE / "outputs/qwen2_5_vl_3b_shared/best_mlp.pt"
BASELINE_METRICS = CODE / "outputs/qwen2_5_vl_3b_shared/metrics.json"

FRAME_ROOT = CODE / "embeddings/qwen2_5_vl_3b_dinov2_base_frames_020"
FRAME_RESULTS = Path("/kaggle/working/outputs/qwen2_5_vl_3b_dinov2_base_frames_020")

RETAIN_RATIO = 0.20
FRAME_ENCODER = "facebook/dinov2-base"

FRAME_RESULTS.mkdir(parents=True, exist_ok=True)

assert CODE.exists(), f"Missing Kaggle code dataset: {CODE}"
for split in ["train", "dev", "test"]:
    assert (FRAME_ROOT / f"meld_{split}.pt").exists(), f"Missing merged {split} embeddings"
assert CHECKPOINT.exists(), f"Missing baseline checkpoint: {CHECKPOINT}"
assert BASELINE_METRICS.exists(), f"Missing baseline metrics: {BASELINE_METRICS}"

print("Code:", CODE)
print("Merged frame embeddings:", FRAME_ROOT)
print("Frame encoder:", FRAME_ENCODER)
print("Frame retain ratio:", RETAIN_RATIO)

In [ ]:
# Only the evaluation/training dependencies are needed; Qwen and DINOv2 are not loaded.
!pip install -q -r {CODE}/requirements.txt

In [ ]:
import json
import pandas as pd
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before MLP evaluation/training.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

for split in ["train", "dev", "test"]:
    payload = torch.load(FRAME_ROOT / f"meld_{split}.pt", map_location="cpu", weights_only=False)
    x = payload["embeddings"].float()
    assert torch.isfinite(x).all(), f"Non-finite embeddings in {split}"
    assert x.shape[0] == payload["labels"].shape[0] == len(payload["sample_ids"])
    print(f"{split}: embeddings={tuple(x.shape)}, errors={len(payload.get('errors', []))}")

## Evaluate the previous MLP

This measures representation shift using the classifier and standardization statistics trained on unpruned Qwen embeddings.

In [ ]:
PREVIOUS_MLP_RESULTS = FRAME_RESULTS / "previous_mlp"

for split in ["dev", "test"]:
    !python {CODE}/scripts/evaluate_mlp.py \
      --checkpoint {CHECKPOINT} \
      --embeddings-pt {FRAME_ROOT}/meld_{split}.pt \
      --output-dir {PREVIOUS_MLP_RESULTS} \
      --split-name {split} \
      --device cuda

## Retrain the MLP on frame-pruned embeddings

The architecture and hyperparameters match the full baseline. Checkpoint selection uses development macro F1.

In [ ]:
NEW_MLP_RESULTS = FRAME_RESULTS / "new_mlp"

!python {CODE}/scripts/train_mlp.py \
  --train-pt {FRAME_ROOT}/meld_train.pt \
  --dev-pt {FRAME_ROOT}/meld_dev.pt \
  --test-pt {FRAME_ROOT}/meld_test.pt \
  --output-dir {NEW_MLP_RESULTS} \
  --hidden-dim 512 \
  --dropout 0.3 \
  --epochs 50 \
  --batch-size 64 \
  --learning-rate 3e-4 \
  --weight-decay 1e-4 \
  --seed 42 \
  --device cuda

## Final performance comparison

In [ ]:
baseline = json.loads(BASELINE_METRICS.read_text())
old_dev = json.loads((PREVIOUS_MLP_RESULTS / "dev_metrics.json").read_text())
old_test = json.loads((PREVIOUS_MLP_RESULTS / "test_metrics.json").read_text())
new = json.loads((NEW_MLP_RESULTS / "metrics.json").read_text())

def metric_row(representation, split, values):
    return {
        "Representation": representation,
        "Split": split,
        "Accuracy": values["accuracy"],
        "Macro F1": values["macro_f1"],
        "Weighted F1": values["weighted_f1"],
    }

rows = [
    metric_row("Full frames + baseline MLP", "Dev", baseline["best_dev"]),
    metric_row("Full frames + baseline MLP", "Test", baseline["test"]),
    metric_row("DINOv2-base diverse 0.20 + previous MLP", "Dev", old_dev),
    metric_row("DINOv2-base diverse 0.20 + previous MLP", "Test", old_test),
    metric_row("DINOv2-base diverse 0.20 + new MLP", "Dev", new["best_dev"]),
    metric_row("DINOv2-base diverse 0.20 + new MLP", "Test", new["test"]),
]
results = pd.DataFrame(rows)
display(results.round(4))

baseline_test = results[(results["Representation"] == "Full frames + baseline MLP") & (results["Split"] == "Test")].iloc[0]
new_test = results[(results["Representation"] == "DINOv2-base diverse 0.20 + new MLP") & (results["Split"] == "Test")].iloc[0]
deltas = pd.DataFrame([{
    "Comparison": "DINOv2-base diverse 0.20 new MLP minus full baseline",
    "Accuracy delta": new_test["Accuracy"] - baseline_test["Accuracy"],
    "Macro F1 delta": new_test["Macro F1"] - baseline_test["Macro F1"],
    "Weighted F1 delta": new_test["Weighted F1"] - baseline_test["Weighted F1"],
}])
display(deltas.round(4))

results.to_csv(FRAME_RESULTS / "metrics_summary.csv", index=False)
deltas.to_csv(FRAME_RESULTS / "baseline_deltas.csv", index=False)

In [ ]:
# Inspect overfitting and identify the selected epoch.
import matplotlib.pyplot as plt

history = pd.read_csv(NEW_MLP_RESULTS / "training_history.csv")
best_epoch = int(history.loc[history["dev_macro_f1"].idxmax(), "epoch"])
print("Best epoch by dev macro F1:", best_epoch)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="train loss")
axes[0].plot(history["epoch"], history["dev_loss"], label="dev loss")
axes[0].axvline(best_epoch, color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(history["epoch"], history["dev_macro_f1"], label="dev macro F1")
axes[1].axvline(best_epoch, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Labeled confusion matrices and per-class reports for the retrained probe.
labels = new.get("label_names", ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"])

def labeled_confusion_matrix(split_metrics):
    return pd.DataFrame(
        split_metrics["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

print("Development confusion matrix")
display(labeled_confusion_matrix(new["best_dev"]))
print("Test confusion matrix")
display(labeled_confusion_matrix(new["test"]))
print("Development classification report")
display(pd.DataFrame(new["best_dev"]["classification_report"]).T.round(4))
print("Test classification report")
display(pd.DataFrame(new["test"]["classification_report"]).T.round(4))

In [ ]:
# Bundle compact result artifacts for download. Embedding tensors remain in FRAME_ROOT.
import shutil

archive_base = Path("/kaggle/working/qwen2_5_vl_3b_dinov2_base_frames_020_results")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=FRAME_RESULTS)
print("Results directory:", FRAME_RESULTS)
print("Embeddings directory:", FRAME_ROOT)
print("Downloadable results archive:", archive_path)
print("Result files:")
for path in sorted(FRAME_RESULTS.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(FRAME_RESULTS))